# Boosting

**Companion lesson:** https://ml-viz.vercel.app/courses/ensemble-methods/02-boosting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## AdaBoost: Sequential Boosting

Each learner focuses on the mistakes of the previous one.

In [ ]:
np.random.seed(42)
n = 80
X = np.sort(5 * np.random.rand(n))
y = np.sign(np.sin(X))  # +1 or -1

def fit_stump(X, y, weights):
    best_loss, best_t, best_p = np.inf, 0, 1
    for t in np.unique(X):
        for p in [1, -1]:
            pred = p * np.sign(X - t)
            pred[pred == 0] = 1
            loss = np.sum(weights * (pred != y))
            if loss < best_loss:
                best_loss, best_t, best_p = loss, t, p
    return best_t, best_p

# AdaBoost
weights = np.ones(n) / n
learners = []
errors = []

for m in range(10):
    t, p = fit_stump(X, y, weights)
    pred = p * np.sign(X - t)
    pred[pred == 0] = 1
    err = np.sum(weights * (pred != y))
    err = np.clip(err, 1e-10, 1 - 1e-10)
    alpha = 0.5 * np.log((1 - err) / err)
    learners.append((t, p, alpha))
    errors.append(err)
    weights *= np.exp(-alpha * y * pred)
    weights /= weights.sum()

x_grid = np.linspace(0, 5, 500)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Show sequential focus
axes[0].scatter(X, y, c=y, cmap='RdYlBu', s=30, alpha=0.5)
for i in range(3):
    t, p, alpha = learners[i]
    pred = p * np.sign(x_grid - t)
    axes[0].plot(x_grid, pred * 0.5 - 1.5 - i * 0.3, color=['#f43f5e', '#eab308', '#14b8a6'][i],
                 linewidth=2, label=f'Round {i+1}')
axes[0].set_title('Each Round Adds a Stump', color='white')
axes[0].legend(fontsize=9)
axes[0].axis('off')

# Final ensemble prediction
final = np.zeros_like(x_grid)
for t, p, alpha in learners:
    final += alpha * (p * np.sign(x_grid - t))
axes[1].scatter(X, y, c='#818cf8', s=20, alpha=0.5)
axes[1].plot(x_grid, np.sign(final), color='#14b8a6', linewidth=2.5, label='Ensemble')
axes[1].plot(x_grid, np.sin(x_grid), '--', color='#94a3b8', label='True')
axes[1].set_title('AdaBoost Ensemble', color='white')
axes[1].legend()
plt.tight_layout()
plt.show()

## One AdaBoost round by hand

The model weight $\alpha=\tfrac12\ln\frac{1-\epsilon}{\epsilon}$ is the minimizer of the exponential loss $(1-\epsilon)e^{-\alpha}+\epsilon e^{\alpha}$. Below we run the lesson's example — 10 points, a stump with 2 errors ($\epsilon=0.2$) — confirm $\alpha=\tfrac12\ln4=0.693$, reweight, and verify the misclassified points end up holding exactly **half** the weight (so the stump's error on the new distribution is 0.5).

In [ ]:
import numpy as np

n = 10
w = np.full(n, 1 / n)                 # uniform weights
# A stump's correctness: True = correct, False = misclassified (2 wrong)
correct = np.array([True]*8 + [False]*2)

eps = w[~correct].sum()               # weighted error
alpha = 0.5 * np.log((1 - eps) / eps)
print(f'epsilon = {eps:.3f}   alpha = 0.5*ln((1-eps)/eps) = {alpha:.3f}  (0.5*ln4 = {0.5*np.log(4):.3f})')

# Reweight: correct *= e^-alpha, wrong *= e^+alpha   (equivalently exp(-alpha * y * h))
factor = np.where(correct, np.exp(-alpha), np.exp(alpha))
w_new = w * factor
print(f'before normalize: correct -> {w_new[correct][0]:.3f} each, wrong -> {w_new[~correct][0]:.3f} each, sum = {w_new.sum():.3f}')
w_new /= w_new.sum()
print(f'after  normalize: correct -> {w_new[correct][0]:.4f} each, wrong -> {w_new[~correct][0]:.4f} each')

# The general AdaBoost identity: the just-trained stump has weighted error 0.5 on the new weights
print(f'\nweight now on the misclassified points = {w_new[~correct].sum():.3f}  (== 0.5, always)')


## Gradient boosting and the learning rate

Boosting fits models **sequentially**, each correcting the previous errors. A smaller learning rate needs more trees but usually generalizes better.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score

X, y = make_classification(n_samples=600, n_features=20, random_state=0)
for lr in [1.0, 0.3, 0.1]:
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, random_state=0)
    s = cross_val_score(gb, X, y, cv=4)
    print(f'learning_rate={lr}: CV accuracy = {s.mean():.3f}')

## Key takeaways

- **Boosting** builds models sequentially, each fixing the previous ensemble's mistakes — cutting **bias**.
- **AdaBoost** reweights misclassified points; **gradient boosting** fits residual gradients.
- **XGBoost / LightGBM** are fast, regularized gradient boosting — top performers on tabular data.
- Lower `learning_rate` + more trees generalizes better; boosting can overfit if unregularized.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — AdaBoost's α and the reweighting step

A weak learner with weighted error $\epsilon$ gets say

$$\alpha = \tfrac{1}{2} \ln\!\frac{1 - \epsilon}{\epsilon}$$

and the sample weights update by $w_i \propto w_i e^{\mp\alpha}$ (down if correct, up if wrong), then renormalize. The last check verifies AdaBoost's beautiful invariant: after reweighting, **the mistakes carry exactly half the total weight** — the learner you just trained becomes useless on the new distribution, forcing the next one to learn something new.

In [ ]:
def adaboost_alpha(err):
    """The say of a weak learner with weighted error err."""
    # TODO(you): 0.5 * ln((1 - err) / err)
    return ...


def reweight(w, correct, alpha):
    """Update and renormalize sample weights. `correct` is a boolean array."""
    w = np.asarray(w, dtype=float)
    correct = np.asarray(correct, dtype=bool)

    # TODO(you): multiply by exp(-alpha) where correct, exp(+alpha) where wrong
    # (hint: np.where(correct, ..., ...))
    w = ...

    # TODO(you): renormalize to sum to 1
    return ...

In [ ]:
# Checks — run me
assert abs(adaboost_alpha(0.5)) < 1e-12, "a coin-flip learner gets zero say"
assert abs(adaboost_alpha(0.1) - 0.5 * np.log(9)) < 1e-12, "err 0.1 -> 0.5 ln 9"
assert adaboost_alpha(0.01) > adaboost_alpha(0.3) > 0, "better learners get more say"

w = np.full(10, 0.1)
correct = np.array([True] * 7 + [False] * 3)   # weighted error = 0.3
w_new = reweight(w, correct, adaboost_alpha(0.3))
assert abs(w_new.sum() - 1) < 1e-12, "weights stay normalized"
assert abs(w_new[~correct].sum() - 0.5) < 1e-12, \
    "after reweighting, the mistakes carry HALF the total weight"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def adaboost_alpha(err):
    return 0.5 * np.log((1 - err) / err)


def reweight(w, correct, alpha):
    w = np.asarray(w, dtype=float)
    correct = np.asarray(correct, dtype=bool)
    w = w * np.exp(np.where(correct, -alpha, alpha))
    return w / w.sum()
```

</details>

### Exercise 2 — Shrinkage geometry

Gradient boosting with a *perfect* weak learner ($h = y - F$) and learning rate $\eta$ shrinks the residual by a factor $(1 - \eta)$ every round:

$$\lVert y - F_k \rVert = (1 - \eta)^k \, \lVert y - F_0 \rVert$$

Implement the loop and verify that exact geometry — it's why $\eta = 1$ nails the training data in one round (and overfits), while small $\eta$ needs many rounds but moves carefully.

In [ ]:
def boost_residual_norm(y, F0, eta, rounds):
    """Norm of the residual after `rounds` of boosting with a perfect weak learner."""
    y = np.asarray(y, dtype=float)
    F = np.full_like(y, float(F0))

    for _ in range(rounds):
        # TODO(you): the perfect weak learner predicts the residual
        h = ...

        # TODO(you): the boosting update F <- F + eta * h
        F = ...

    return float(np.linalg.norm(y - F))

In [ ]:
# Checks — run me
y = np.array([3.0, -1.0, 2.0])
r0 = np.linalg.norm(y - 0.0)

assert abs(boost_residual_norm(y, 0.0, 1.0, 1)) < 1e-12, "eta = 1 + perfect learner: one round nails it"
assert abs(boost_residual_norm(y, 0.0, 0.1, 5) - (0.9 ** 5) * r0) < 1e-9, \
    "each round shrinks the residual by (1 - eta)"
assert boost_residual_norm(y, 0.0, 0.05, 50) > 1e-12, "small eta needs many rounds — the shrinkage trade-off"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def boost_residual_norm(y, F0, eta, rounds):
    y = np.asarray(y, dtype=float)
    F = np.full_like(y, float(F0))
    for _ in range(rounds):
        h = y - F
        F = F + eta * h
    return float(np.linalg.norm(y - F))
```

</details>